# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravindidhananjana/Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row represents one content item (content_hash_id) for a specific client (client_hash_id).
Time Window: Mid-panel month of March 2026 (report_date between 2026-03-01 and 2026-03-31).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify single mid-panel month date span and row counts
import os, getpass
import duckdb

# 1. Get Hugging Face Token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

# 2. Connect DuckDB & Register Secret
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# 3. Define Table Paths
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':        f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':        f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':         f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':  f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':     f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connection established and tables configured successfully!")

contract_span = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS total_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print(contract_span)


Paste your Hugging Face READ token (hf_...): ··········
DuckDB connection established and tables configured successfully!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  start_date   end_date  total_rows
0 2026-03-01 2026-03-31     9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: imp_prev15 (15-day prior impressions), pos_avg_prev (15-day prior avg position), pos_std_prev (position volatility/variance), visible_queries (query coverage count), top_query_share (query concentration share).Label: is_declining — Binary flag (1 if impressions in the second half of the month drop by $>20\%$ compared to the first half, else 0).Context: client_hash_id, content_hash_id.Excluded: trend_direction and trend_pct — Excluded because they are calculated directly from future performance metrics (deliberate target leakage).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify column names and check for missing values across selected context and feature fields
fields_check = con.sql(f"""
    SELECT
        COUNT(client_hash_id) AS client_id_count,
        COUNT(content_hash_id) AS content_id_count,
        COUNT(gsc_impressions) AS imp_count,
        COUNT(gsc_avg_position) AS pos_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print(fields_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   client_id_count  content_id_count  imp_count  pos_count
0          9841378           9841378    9841378    3611061


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Query 1: Grain Probe (Must yield 0 rows) ---
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS cnt
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

print("1. Grain Probe duplicate count (should be 0):", len(grain_check))


# --- Query 2: Slice Row Count & Date Span ---
slice_info = con.sql(f"""
    SELECT
        COUNT(*) AS total_daily_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print("\n2. Slice Summary:")
print(slice_info)


# --- Query 3: Availability Filter Check ---
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(CASE WHEN ga4_data_available = TRUE THEN 1 END) AS valid_ga4_rows,
        ROUND(COUNT(CASE WHEN ga4_data_available = TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS ga4_availability_pct
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print("\n3. GA4 Availability Check:")
print(availability_check)


# --- Build 5 Features & Merge Query Signals ---
# Features knowable at decision moment (March 15, 2026):
# 1. imp_prev15: Historical impression volume up to decision moment.
# 2. pos_avg_prev: Average rank during the prior 15-day observation period.
# 3. pos_std_prev: Search rank stability/volatility prior to decision moment.
features_df = con.sql(f"""
    WITH windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Outcome Window (March 16-31)
            SUM(CASE WHEN f.report_date > DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_last15,

            -- Historical Window (March 1-15)
            SUM(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_prev15,
            AVG(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_avg_position END) AS pos_avg_prev,
            STDDEV_SAMP(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_avg_position END) AS pos_std_prev

        FROM {TABLES['fact_daily']} f
        WHERE f.report_date >= '2026-03-01' AND f.report_date <= '2026-03-31'
        GROUP BY 1, 2
        HAVING imp_prev15 >= 10
    )
    SELECT * FROM windowed
""").df().fillna({'pos_std_prev': 0})

# Load Query Signals for Features 4 and 5:
# 4. visible_queries: Total distinct search queries driving traffic (knowable from 90d query log).
# 5. top_query_share: Share of traffic coming from top query (knowable from query distribution).
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           MAX(impressions_90d) / NULLIF(SUM(impressions_90d), 0) AS top_query_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

features_df = features_df.merge(qsignals, on='content_hash_id', how='left').fillna(0)
features_df['is_declining'] = (features_df['imp_last15'] < 0.8 * features_df['imp_prev15']).astype(int)


# --- Deliberate Leakage Trap Experiment ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Inject explicit leakage column (future outcome traffic used directly as a feature)
features_df['leaked_future_signal'] = features_df['imp_last15']

X_leaked = features_df[['imp_prev15', 'pos_avg_prev', 'leaked_future_signal']]
y_leaked = features_df['is_declining']

model_leak = RandomForestClassifier(random_state=42).fit(X_leaked, y_leaked)
acc_leak = accuracy_score(y_leaked, model_leak.predict(X_leaked))
print(f"\n[LEAK TRAP] Accuracy with leaked feature: {acc_leak:.4f}")

# Clean features without leakage
X_honest = features_df[['imp_prev15', 'pos_avg_prev', 'pos_std_prev', 'visible_queries', 'top_query_share']]
y_honest = features_df['is_declining']

model_honest = RandomForestClassifier(random_state=42).fit(X_honest, y_honest)
acc_honest = accuracy_score(y_honest, model_honest.predict(X_honest))
print(f"[HONEST EVAL] Accuracy without leaked feature: {acc_honest:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Grain Probe duplicate count (should be 0): 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


2. Slice Summary:
   total_daily_rows  distinct_content_items   min_date   max_date
0           9841378                  331437 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


3. GA4 Availability Check:
   total_rows  valid_ga4_rows  ga4_availability_pct
0     9841378          413966                  4.21


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


[LEAK TRAP] Accuracy with leaked feature: 1.0000
[HONEST EVAL] Accuracy without leaked feature: 0.9999


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitation: Unbalanced Client Panels and GA4 Coverage Gap. GSC tracking start dates vary significantly across client domains, and only 4.21% of daily records have active GA4 analytics (ga4_data_available = TRUE). This restricts joint search-and-behavioral modeling to a small subset of clients.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.